In [1]:
import pandas as pd
import numpy as np 
import sys 
sys.path.append('../utils/')
from mxnet_utils import *

In [2]:
test_dataset_raw = load_tsv_data('../data/liar-plus/test2.tsv')

def analyze_dataset(raw_data):
    print("analyzing raw data set...\n <><><><><><><><><><> \n")

    label_cnt = Counter()
    topic_cnt = Counter()
    author_cnt = Counter()
    job_cnt = Counter()
    location_cnt = Counter()
    affiliation_cnt = Counter()

    # Try to extract the valid labels, topics, authors,
    for ele in raw_data:
        # Ensure element has enough columns
        if len(ele) < 13:
            continue

        try:
            label, statement, topics, author, job, location, affiliation,cnt_barely, cnt_false, cnt_half, cnt_mostly, cnt_pants_on_fire, venue_context, justification = [str(e) for e in ele[2:16]]

            label_cnt[label] += 1
            for topic in topics.lower().split(','):
                topic_cnt[topic.strip()] += 1
            author_cnt[author.lower()] += 1
            job_cnt[job.lower()] += 1
            location_cnt[location.lower()] += 1
            affiliation_cnt[affiliation.lower()] += 1
        except Exception as e:
            print(f"Error processing row: {ele} | Error: {e}")

    canonical = ["false","barely-true","half-true","mostly-true","true","pants-fire"]
    label_map = {lab:i for i,lab in enumerate(canonical)}
    print("label map:",label_map)

    #!consider tuning the min_freq param
    # Replace gluonnlp.Vocab with CustomVocab
    topic_vocab = CustomVocab(topic_cnt, min_freq=100)
    author_vocab = CustomVocab(author_cnt, min_freq=50)
    job_vocab = CustomVocab(job_cnt, min_freq=50)
    location_vocab = CustomVocab(location_cnt, min_freq=50)
    affiliation_vocab = CustomVocab(affiliation_cnt, min_freq=50)

    print(topic_vocab)
    print(author_vocab)
    print(job_vocab)
    print(location_vocab)
    print(affiliation_vocab)
    return topic_vocab, author_vocab, job_vocab, location_vocab, affiliation_vocab, label_map

def feature_extraction_transform(data):
    # Transform label into position / negative
    try:
        label, statement, topics, author, job, location, affiliation,cnt_barely, cnt_false, cnt_half, cnt_mostly, cnt_pants_on_fire, venue_context, justification = [str(e) for e in data[2:16]]
    except Exception as e:
        print(f"Error in feature_extraction_transform: {e}, data: {data}")
        return None # Return None to filter out this bad data

    topic_one_encoding = np.zeros(shape=(len(topic_vocab)), dtype=np.float32)
    topic_ids = [topic_vocab[t.strip()] for t in topics.lower().split(',')]
    topic_one_encoding[topic_ids] = 1
    if len(topic_ids) > 0 and topic_one_encoding.sum() > 0:
        topic_one_encoding /= topic_one_encoding.sum()

    author_id = author_vocab[author.lower()]
    job_id = job_vocab[job.lower()]
    location_id = location_vocab[location.lower()]
    affiliation_id = affiliation_vocab[affiliation.lower()]

    try:
        cnt_barely_f = float(cnt_barely)
        cnt_false_f = float(cnt_false)
        cnt_half_f = float(cnt_half)
        cnt_mostly_f = float(cnt_mostly)
        cnt_pants_on_fire_f = float(cnt_pants_on_fire)
    except ValueError:
        # Handle cases where conversion to float fails
        cnt_barely_f = cnt_false_f = cnt_half_f = cnt_mostly_f = cnt_pants_on_fire_f = 0.0

    cnt_total = cnt_barely_f + cnt_false_f + cnt_half_f + cnt_mostly_f + cnt_pants_on_fire_f

    if cnt_total > 0 :
        proportion = [cnt_barely_f / cnt_total,
                      cnt_false_f / cnt_total,
                      cnt_half_f / cnt_total,
                      cnt_mostly_f / cnt_total,
                      cnt_pants_on_fire_f / cnt_total]
    else:
        proportion = [0.0, 0.0, 0.0, 0.0, 0.0]

    cnt_uncertainty = 1.0 / (cnt_total + 1.0)
    history_of_truth = np.array(proportion + [cnt_uncertainty], dtype=np.float32)
    venue_feature = 0 # venue_feature = f(venue_context), keep it as a vector

    return (statement, topic_one_encoding, author_id, job_id, location_id, affiliation_id,
            history_of_truth, venue_feature, label_map[label])

In [3]:
import pickle

with open("../checkpoints/vocabs.pkl", "rb") as f:
    vocabs = pickle.load(f)

topic_vocab       = vocabs["topic_vocab"]
author_vocab      = vocabs["author_vocab"]
job_vocab         = vocabs["job_vocab"]
location_vocab    = vocabs["location_vocab"]
affiliation_vocab = vocabs["affiliation_vocab"]
label_map         = vocabs["label_map"]

print("Loaded vocabs and label_map.")

Loaded vocabs and label_map.


In [4]:
test_dataset = [d for d in [feature_extraction_transform(data) for data in test_dataset_raw] if d is not None]
test_dataset_bert = [transform_fn(*sample) for sample in test_dataset]
test_data_torch = ListDataset(test_dataset_bert)
test_data = DataLoader(test_data_torch,
                       batch_size=16,
                       shuffle=False,
                       collate_fn=collate_fn,
                       num_workers=0,
                       pin_memory=False)

In [5]:
import torch
import random
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from transformers import BertModel, BertTokenizer, BertConfig

np.random.seed(100)
torch.manual_seed(100)
random.seed(100)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(100)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

bert_dropout = 0.1 # From later in the notebook
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
config = BertConfig.from_pretrained('bert-base-uncased',
                                    hidden_dropout_prob=bert_dropout,
                                    attention_probs_dropout_prob=bert_dropout)
bert_base = BertModel.from_pretrained('bert-base-uncased', config=config)
bert_base.to(device)
print(bert_base)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [6]:
import torch
from torch.nn.functional import softmax

# -------------------------------------------------------
# 1. Rebuild model exactly as during training
# -------------------------------------------------------
model = BERTClassifier(
    bert=bert_base,
    num_topics=len(topic_vocab),
    num_authors=len(author_vocab),
    num_jobs=len(job_vocab),
    num_locations=len(location_vocab),
    num_affiliations=len(affiliation_vocab),
    num_classes=len(label_map),
    embed_dim=32,
    author_dropout=0.1,
    author_mlp_layers=2,
    author_mlp_hidden=192,
    history_dropout=0.1,
    history_mlp_layers=2,
    history_mlp_hidden=192
).to(device)

model.load_state_dict(torch.load("../checkpoints/best.pth", map_location=device))
model.eval()

print("Loaded best model.")

# -------------------------------------------------------
# 2. Run predictions on test_data
# -------------------------------------------------------
all_preds = []
all_probs = []

with torch.no_grad():
    for (
        inputs, segment_types, attention_mask,
        topic_one_hot, author_id, job_id, location_id,
        affiliation_id, history_feature, labels
    ) in test_dataset_raw:

        inputs = inputs.to(device)
        segment_types = segment_types.to(device)
        attention_mask = attention_mask.to(device)
        topic_one_hot = topic_one_hot.to(device)
        author_id = author_id.to(device)
        job_id = job_id.to(device)
        location_id = location_id.to(device)
        affiliation_id = affiliation_id.to(device)
        history_feature = history_feature.to(device)

        logits = model(
            inputs, segment_types, attention_mask,
            topic_one_hot, author_id, job_id, location_id,
            affiliation_id, history_feature
        )

        probs = softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

print("Prediction shape:", all_probs.shape)

# -------------------------------------------------------
# 3. Attach predictions to test dataframe
# -------------------------------------------------------
# You loaded the raw TSV earlier as: test_dataset_raw
# Convert that to a DataFrame to attach predictions
test_df = pd.DataFrame(test_dataset_raw)

# Add columns
test_df["pred_label"] = all_preds
for i in range(all_probs.shape[1]):
    test_df[f"prob_{i}"] = all_probs[:, i]

print(test_df.head())

# Now test_df contains:
# - pred_label
# - prob_0 ... prob_5


Loaded best model.


ValueError: too many values to unpack (expected 10)